# Train court-keypoint model (33 pts) — CLEAN data + heavy appearance augmentation

**Goal:** cross-arena robustness *without* out-of-domain data (the external-merge path failed 3×). Trains on the **clean original court_det2 (850 imgs, in-domain broadcast)** with **heavy HSV augmentation** (hsv_h 0.03 / hsv_s 0.9 / hsv_v 0.7) so the model becomes invariant to arena floor-color, lighting, and broadcast color grading.

**Before running:** GPU runtime (T4), and upload the new `court_det2_colab.zip` (59 MB, clean) to My Drive root.

`epochs=300` ceiling, `patience=30` (heavy aug plateaus slower). Checkpoints -> Drive so a disconnect is recoverable via **cell 5**.

**Cells:** 1 GPU → 2 mount+unzip (expect 610/124/116) → 3 train → 4 confirm weights → 5 resume.

When done: download `court_aug/best.pt` → replace `models/court_kp33.pt` (old backed up as `court_kp33_pre_merge.pt`).

In [1]:
# 1) Confirm GPU
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "— NO GPU, set Runtime → GPU")

CUDA: True Tesla T4


In [ ]:
# 2) Load the CLEAN dataset from Google Drive (original court_det2 only, for the appearance-aug run).
#    files.upload() does NOT work in the Cursor extension, so we mount Drive instead.
#    FIRST (in a browser): upload the new court_det2_colab.zip to the TOP LEVEL of My Drive.
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')
ZIP = "/content/drive/MyDrive/court_det2_colab.zip"
import os; assert os.path.exists(ZIP), f"Not found: {ZIP}  -> upload the zip to My Drive root first"
!rm -rf /content/court_det2 && mkdir -p /content/court_det2
!unzip -q "$ZIP" -d /content/court_det2
ROOT = "/content/court_det2"
print(open(f"{ROOT}/data_colab.yaml").read())          # confirm kpt_shape: [33, 3]
import glob
print("train imgs:", len(glob.glob(f"{ROOT}/train/images/*")),
      "| valid:", len(glob.glob(f"{ROOT}/valid/images/*")),
      "| test:", len(glob.glob(f"{ROOT}/test/images/*")))   # expect 610 / 124 / 116 (clean original)

In [ ]:
# 3) Train yolov8m-POSE on CLEAN original court_det2 (850 imgs) + HEAVY APPEARANCE AUGMENTATION.
#    hsv_* cranked up so the model becomes invariant to arena floor-color / lighting / broadcast grading
#    (cross-arena robustness) WITHOUT adding out-of-domain data. Geometry aug unchanged.
from ultralytics import YOLO
YOLO("yolov8m-pose.pt").train(
    data=f"{ROOT}/data_colab.yaml", epochs=300, imgsz=640, batch=16, device=0,
    project="/content/drive/MyDrive/court_kp_runs", name="court_aug", patience=30,
    fliplr=0.5, degrees=0.0, translate=0.05, scale=0.2, mosaic=0.0,   # geometry (unchanged)
    hsv_h=0.03, hsv_s=0.9, hsv_v=0.7)                                 # HEAVY color/brightness aug
print("\nDONE -> /content/drive/MyDrive/court_kp_runs/court_aug/weights/best.pt")

In [ ]:
# 4) Weights are ALREADY on Drive (project pointed there). Confirm + note where to put them.
import glob
print("best.pt:", glob.glob("/content/drive/MyDrive/court_kp_runs/court_aug*/weights/best.pt"))
print("last.pt:", glob.glob("/content/drive/MyDrive/court_kp_runs/court_aug*/weights/last.pt"))
print("\n-> download best.pt, replace models/court_kp33.pt (old backed up as court_kp33_pre_merge.pt)")

In [ ]:
# 5) ===== RESUME AFTER A DISCONNECT =====
# Reconnect to a GPU runtime, then run ONLY this cell (not 2-3). Re-mounts Drive, reinstalls
# ultralytics, re-unzips the dataset (/content was wiped), resumes the SAME run from last.pt.
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists("/content/court_det2/data_colab.yaml"):
    !rm -rf /content/court_det2 && mkdir -p /content/court_det2
    !unzip -q "/content/drive/MyDrive/court_det2_colab.zip" -d /content/court_det2
LAST = "/content/drive/MyDrive/court_kp_runs/court_aug/weights/last.pt"
assert os.path.exists(LAST), f"No checkpoint at {LAST} (did cell 3 run long enough to save one?)"
from ultralytics import YOLO
YOLO(LAST).train(resume=True)
print("\nresumed -> continuing toward epoch 300")